In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

import warnings
warnings.filterwarnings('ignore')

In [28]:
train = '/kaggle/input/competitions/playground-series-s6e4/train.csv'
test = '/kaggle/input/competitions/playground-series-s6e4/test.csv'
sub_sample = '/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv'
original = '/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv'

train_df = pd.read_csv(train).dropna(subset=['Irrigation_Need'])
test_df = pd.read_csv(test)
original_df = pd.read_csv(original)

print("train: shape: ", train_df.shape)
print("test shape: ", test_df.shape)
print("original shape: ", original_df.shape)

original_df['id'] = range(train_df['id'].max() + 1, train_df['id'].max() + 1 + len(original_df))

train_full = pd.concat([train_df, original_df], axis=0, ignore_index=True)

print("train_full shape: ", train_full.shape)

train: shape:  (630000, 21)
test shape:  (270000, 20)
original shape:  (10000, 20)
train_full shape:  (640000, 21)


In [29]:
train_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640000 entries, 0 to 639999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       640000 non-null  int64  
 1   Soil_Type                640000 non-null  object 
 2   Soil_pH                  640000 non-null  float64
 3   Soil_Moisture            640000 non-null  float64
 4   Organic_Carbon           640000 non-null  float64
 5   Electrical_Conductivity  640000 non-null  float64
 6   Temperature_C            640000 non-null  float64
 7   Humidity                 640000 non-null  float64
 8   Rainfall_mm              640000 non-null  float64
 9   Sunlight_Hours           640000 non-null  float64
 10  Wind_Speed_kmh           640000 non-null  float64
 11  Crop_Type                640000 non-null  object 
 12  Crop_Growth_Stage        640000 non-null  object 
 13  Season                   640000 non-null  object 
 14  Irri

In [30]:
cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

num_cols = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
            'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
            'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

In [31]:
def engineer_features(df, moisture_map=None):
    """Create domain-driven interaction features."""
    df = df.copy()
    
    # Water balance features
    df['Total_Water_Input'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    df['Evaporative_Stress'] = (df['Temperature_C'] * df['Sunlight_Hours'] * df['Wind_Speed_kmh']) / (df['Humidity'] + 1)
    df['Moisture_Deficit'] = df['Humidity'] - df['Soil_Moisture']
    df['Water_Balance'] = df['Total_Water_Input'] - df['Evaporative_Stress'] * df['Field_Area_hectare']
    
    # Crop-relative moisture (target encoding proxy - no leakage)
    if moisture_map is None:
        moisture_map = df.groupby('Crop_Type')['Soil_Moisture'].mean()
    df['Relative_Crop_Moisture'] = df['Soil_Moisture'] / df['Crop_Type'].map(moisture_map).fillna(1.0)
    
    # Mulch interaction
    df['Mulch_Moisture'] = df['Soil_Moisture'] * df['Mulching_Used'].map({'Yes': 1.5, 'No': 1.0}).fillna(1.0)
    
    # Soil health features
    df['Soil_Health'] = df['Organic_Carbon'] * df['Soil_Moisture'] / (df['Electrical_Conductivity'] + 0.1)
    df['Soil_Salinity_Risk'] = df['Electrical_Conductivity'] * df['Temperature_C'] / (df['Rainfall_mm'] + 1)
    df['pH_Deviation'] = np.abs(df['Soil_pH'] - 6.5)
    
    # Climate stress features
    df['Heat_Stress'] = df['Temperature_C'] * (100 - df['Humidity']) / 100
    df['Dryness_Index'] = df['Temperature_C'] * df['Sunlight_Hours'] / (df['Rainfall_mm'] + 1)
    df['Wind_Evap'] = df['Wind_Speed_kmh'] * df['Temperature_C'] / (df['Humidity'] + 1)
    
    # Field efficiency
    df['Irrigation_Per_Hectare'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 0.1)
    df['Rainfall_Per_Hectare'] = df['Rainfall_mm'] / (df['Field_Area_hectare'] + 0.1)
    
    # Moisture retention
    df['Moisture_Retention'] = df['Soil_Moisture'] * df['Organic_Carbon']
    df['Moisture_Temp_Ratio'] = df['Soil_Moisture'] / (df['Temperature_C'] + 1)
    
    # Crop-soil-season interactions (encoded as grouped stats)
    for grp_col in ['Soil_Type', 'Season', 'Crop_Growth_Stage']:
        grp_key = f'{grp_col}_Moisture_mean'
        if moisture_map is not None and grp_key in moisture_map.index.names:
            pass  # skip if already computed
        grp = df.groupby(grp_col)['Soil_Moisture'].transform('mean')
        df[f'{grp_col}_Moisture_dev'] = df['Soil_Moisture'] - grp
    
    return df, moisture_map

train_full, moisture_map = engineer_features(train_full)
test_df, _ = engineer_features(test_df, moisture_map)

feature_cols = [c for c in train_full.columns if c not in ['id', 'Irrigation_Need'] and train_full[c].dtype != 'object']
# Add back cat cols
feature_cols = cat_cols + [c for c in feature_cols if c not in cat_cols]

print(f'Total features: {len(feature_cols)}')
print(f'Engineered features: {[c for c in feature_cols if c not in cat_cols + num_cols]}')

Total features: 38
Engineered features: ['Total_Water_Input', 'Evaporative_Stress', 'Moisture_Deficit', 'Water_Balance', 'Relative_Crop_Moisture', 'Mulch_Moisture', 'Soil_Health', 'Soil_Salinity_Risk', 'pH_Deviation', 'Heat_Stress', 'Dryness_Index', 'Wind_Evap', 'Irrigation_Per_Hectare', 'Rainfall_Per_Hectare', 'Moisture_Retention', 'Moisture_Temp_Ratio', 'Soil_Type_Moisture_dev', 'Season_Moisture_dev', 'Crop_Growth_Stage_Moisture_dev']


In [32]:
target_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
reverse_mapping = {0: 'Low', 1: 'Medium', 2: 'High'}
y = train_full['Irrigation_Need'].map(target_mapping).values

for col in cat_cols:
    combined_cats = pd.concat([train_full[col], test_df[col]]).astype(str).unique()
    train_full[col] = pd.Categorical(train_full[col].astype(str), categories=combined_cats)
    test_df[col] = pd.Categorical(test_df[col].astype(str), categories=combined_cats)

X_train = train_full[feature_cols].copy()
X_test = test_df[feature_cols].copy()
test_ids = test_df['id'].values


for c in cat_cols:
    X_train[c] = X_train[c].astype('category')
    X_test[c] = X_test[c].astype('category')

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (640000, 38), X_test: (270000, 38)


In [33]:
lgb_params = dict(device='gpu', verbose=-1)

xgb_params = dict(device='cuda', tree_method='hist', enable_categorical=True)

cat_params = dict(task_type='GPU', auto_class_weights='Balanced', cat_features=cat_cols, verbose=0)

model_configs = [
    ('lgb', lgb.LGBMClassifier, lgb_params, False),
    ('xgb', xgb.XGBClassifier, xgb_params, True),
    ('cat', cb.CatBoostClassifier, cat_params, False),
]

print(f'{len(model_configs)}')

3


In [34]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sample_weights = compute_sample_weight('balanced', y)

n_models = len(model_configs)
oof_preds = {name: np.zeros((len(X_train), 3)) for name, _, _, _ in model_configs}
test_preds = {name: np.zeros((len(X_test), 3)) for name, _, _, _ in model_configs}


for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    print(f'\n=== Fold {fold+1}/{5} ===')
    xt, xv = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    yt, yv = y[tr_idx], y[val_idx]
    wt = sample_weights[tr_idx]
    
    for name, ModelClass, params, use_sample_weight in model_configs:
        model = ModelClass(**params)
        
        if use_sample_weight:
            model.fit(xt, yt, sample_weight=wt)
        else:
            model.fit(xt, yt)
        
        oof_preds[name][val_idx] = model.predict_proba(xv)
        test_preds[name] += model.predict_proba(X_test) / 5
        
        score = balanced_accuracy_score(yv, oof_preds[name][val_idx].argmax(1))
        print(f'  {name}: {score:.5f}')

print(f'\n=== Overall OOF Balanced Accuracy ===')
for name in oof_preds:
    score = balanced_accuracy_score(y, oof_preds[name].argmax(1))
    print(f'  {name}: {score:.5f}')


=== Fold 1/5 ===


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  lgb: 0.96351
  xgb: 0.96950
  cat: 0.96847

=== Fold 2/5 ===
  lgb: 0.96354
  xgb: 0.96948
  cat: 0.96740

=== Fold 3/5 ===
  lgb: 0.96094
  xgb: 0.96972
  cat: 0.96769

=== Fold 4/5 ===
  lgb: 0.96284
  xgb: 0.97094
  cat: 0.96878

=== Fold 5/5 ===
  lgb: 0.95805
  xgb: 0.96644
  cat: 0.96344

=== Overall OOF Balanced Accuracy ===
  lgb: 0.96177
  xgb: 0.96922
  cat: 0.96716


In [38]:
oof_stack = np.hstack([oof_preds[name] for name in oof_preds])
test_stack = np.hstack([test_preds[name] for name in test_preds])
print(f'Meta-features: {oof_stack.shape[1]} (3 probs x {n_models} models)')

avg_oof = np.mean([oof_preds[name] for name in oof_preds], axis=0)
avg_test = np.mean([test_preds[name] for name in test_preds], axis=0)
avg_score = balanced_accuracy_score(y, avg_oof.argmax(1))
print(f'Simple average OOF balanced accuracy: {avg_score:.5f}')

meta_oof_preds = np.zeros(len(y), dtype=int)
meta_test_probs = np.zeros((len(X_test), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    meta = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0, random_state=42)
    meta.fit(oof_stack[tr_idx], y[tr_idx])
    meta_oof_preds[val_idx] = meta.predict(oof_stack[val_idx])
    meta_test_probs += meta.predict_proba(test_stack) / 5

stack_score = balanced_accuracy_score(y, meta_oof_preds)
print(f'Stacked meta-learner OOF balanced accuracy: {stack_score:.5f}')

final_meta = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0, random_state=42)
final_meta.fit(oof_stack, y)
final_meta_test = final_meta.predict_proba(test_stack)

if stack_score >= avg_score:
    print('\n>>> Using stacked meta-learner predictions')
    final_test_probs = final_meta_test
    best_oof_probs = np.zeros((len(y), 3))
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
        meta = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0, random_state=42)
        meta.fit(oof_stack[tr_idx], y[tr_idx])
        best_oof_probs[val_idx] = meta.predict_proba(oof_stack[val_idx])
else:
    print('\n>>> Using simple average predictions')
    final_test_probs = avg_test
    best_oof_probs = avg_oof

Meta-features: 9 (3 probs x 3 models)
Simple average OOF balanced accuracy: 0.96675
Stacked meta-learner OOF balanced accuracy: 0.97116

>>> Using stacked meta-learner predictions


In [40]:
final_adjusted = final_test_probs
final_preds = final_adjusted.argmax(axis=1)
final_labels = [reverse_mapping[p] for p in final_preds]

submission = pd.DataFrame({'id': test_ids, 'Irrigation_Need': final_labels})
submission.to_csv('submission.csv', index=False)

print('Submission shape:', submission.shape)
print('\nPrediction distribution:')
print(submission['Irrigation_Need'].value_counts(normalize=True))
print('\nFirst 10 rows:')
print(submission.head(10))
print('\nSubmission saved!')

Submission shape: (270000, 2)

Prediction distribution:
Irrigation_Need
Low       0.591800
Medium    0.371711
High      0.036489
Name: proportion, dtype: float64

First 10 rows:
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
5  630005          Medium
6  630006             Low
7  630007          Medium
8  630008            High
9  630009             Low

Submission saved!
